In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.preprocessing import StandardScaler

from src.analysis.dim_reducer import reduce_dimensionality
from src.util.datasets import load_mnist_dataset

DATASET_PATH = Path("datasets/wine_quality/wine+quality/winequality-red.csv")
df = pd.read_csv(DATASET_PATH, sep=";")
df = df.reset_index(drop=True)
df["row_id"] = df.index

/Users/hannesmoehring/Documents/University/SEM8/SHD/dev/SHD/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X = df.drop(columns=["quality", "row_id"]).values
scaler = StandardScaler()
X_scaled: pd.DataFrame = scaler.fit_transform(X)

In [3]:
mnist_data = load_mnist_dataset()
# mnist_x_combined = np.concatenate([mnist_data[0], mnist_data[2]], axis=0)
mnist_x_combined = mnist_data[0]  # only use training data for now
mnist_df = pd.DataFrame(mnist_x_combined, columns=[f"pixel_{i}" for i in range(784)])
# mnist_df["label"] = mnist_data[1] + mnist_data[3]
mnist_df["label"] = mnist_data[1]
mnist_df["row_id"] = mnist_df.index
mnist_x_scaled = scaler.fit_transform(mnist_x_combined)
mnist_X = mnist_df.drop(columns=["label", "row_id"]).values


In [ ]:
import umap
reducer = umap.UMAP(n_components=10, n_neighbors=30, min_dist=0.0)
X_umap = reducer.fit_transform(X_scaled)
model = HDBSCAN(min_cluster_size=15, min_samples=5)
labels = model.fit_predict(X_umap)
df["cluster"] = labels

/Users/hannesmoehring/Documents/University/SEM8/SHD/dev/SHD/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,row_id,cluster
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,0,10
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,1,20
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,2,20
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,3,9
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,4,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5,1594,6
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6,1595,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6,1596,6
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5,1597,6


In [ ]:
import umap
from sklearn.manifold import MDS

reducer = umap.UMAP(n_components=10, n_neighbors=30, min_dist=0.0)
X_umap = reducer.fit_transform(X_scaled)
model = HDBSCAN(min_cluster_size=15, min_samples=5)
labels = model.fit_predict(X_umap)
df["cluster"] = labels

# exclude HDBSCAN noise points
mask = df["cluster"] != -1
feature_cols = df.columns.drop(["quality", "row_id", "cluster"])

X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols, index=df.index)
centroids = X_scaled_df[mask].groupby(df.loc[mask, "cluster"]).mean()

# 2D layout that preserves pairwise centroid distances
mds = MDS(n_components=2, dissimilarity="euclidean",
          random_state=42, n_init=8, normalized_stress="auto")
centroids_2d = mds.fit_transform(centroids.values)

sizes = df.loc[mask, "cluster"].value_counts().sort_index()

layout_df = pd.DataFrame({
    "x": centroids_2d[:, 0],
    "y": centroids_2d[:, 1],
    "cluster": centroids.index,
    "size": sizes.values,
})


fig = px.scatter(layout_df, x="x", y="y", size="size", color="size", hover_data=["cluster"],
                 title="HDBSCAN Clusters of Wine Quality Dataset (MDS Layout)")

fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(legend_title_text="Cluster")
fig.show()

/Users/hannesmoehring/Documents/University/SEM8/SHD/dev/SHD/.venv/lib/python3.13/site-packages/sklearn/manifold/_mds.py:754: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
/Users/hannesmoehring/Documents/University/SEM8/SHD/dev/SHD/.venv/lib/python3.13/site-packages/sklearn/manifold/_mds.py:771: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(


In [13]:
from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde

fig = go.Figure()

for i, c in enumerate(centroids.index):
    pts = X_scaled_df.loc[df["cluster"] == c].values
    if len(pts) < 5:
        continue

    # local 2D embedding of just this cluster
    pca = PCA(n_components=2)
    pts_2d = pca.fit_transform(pts)

    # KDE on the local 2D points
    kde = gaussian_kde(pts_2d.T, bw_method="scott")
    pad = 0.5 * pts_2d.std(axis=0).max()
    lx_min, lx_max = pts_2d[:, 0].min() - pad, pts_2d[:, 0].max() + pad
    ly_min, ly_max = pts_2d[:, 1].min() - pad, pts_2d[:, 1].max() + pad
    lx = np.linspace(lx_min, lx_max, 60)
    ly = np.linspace(ly_min, ly_max, 60)
    LX, LY = np.meshgrid(lx, ly)
    Z = kde(np.vstack([LX.ravel(), LY.ravel()])).reshape(LX.shape)

    # scale the local footprint and place at the cluster's MDS position
    cx, cy = centroids_2d[i]
    local_extent = max(lx_max - lx_min, ly_max - ly_min)
    target_size = 0.8 * np.sqrt(sizes.loc[c] / sizes.max()) + 0.3  # tunable
    scale = target_size / local_extent

    plot_x = (lx - (lx_min + lx_max) / 2) * scale + cx
    plot_y = (ly - (ly_min + ly_max) / 2) * scale + cy

    fig.add_trace(go.Contour(
        x=plot_x, y=plot_y, z=Z,
        colorscale="Viridis", showscale=False,
        contours=dict(coloring="heatmap", showlines=True,
                      start=Z.max() * 0.05, size=Z.max() * 0.15),
        line_smoothing=0.85, hoverinfo="skip",
    ))

fig.add_trace(go.Scatter(
    x=centroids_2d[:, 0], y=centroids_2d[:, 1],
    mode="text",
    text=[f"C{c}" for c in centroids.index],
    textposition="top center",
    showlegend=False,
))

fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(
    title="Cluster topography — global MDS layout, local KDE per cluster",
    plot_bgcolor="white",
)
fig.show()

In [14]:
from sklearn.tree import DecisionTreeClassifier, export_text
from plotly.subplots import make_subplots
import plotly.graph_objects as go

feature_cols = df.columns.drop(["quality", "row_id", "cluster"])

def cluster_characteristics(cluster_id, df, X_scaled_df, feature_cols,
                            top_n=8, tree_depth=3):
    in_cluster = df["cluster"] == cluster_id
    pts = X_scaled_df.loc[in_cluster, feature_cols]

    # in scaled space, global mean=0 and global std=1, so:
    z_mean = pts.mean()   # signed z-score of cluster mean per dim
    z_std  = pts.std()    # within-cluster std, in units of global std
    order = z_mean.abs().sort_values(ascending=False).index.tolist()
    z_mean, z_std = z_mean[order], z_std[order]

    R = max(3.0, z_mean.abs().max() + 1)   # radial extent in z-units

    fig = make_subplots(rows=1, cols=2,
        specs=[[{"type": "polar"}, {"type": "xy"}]],
        column_widths=[0.55, 0.45],
        subplot_titles=(f"Cluster {cluster_id} profile",
                        "Top distinguishing dimensions"))

    theta = order + [order[0]]

    # reference ring at global mean (z=0 → r=R)
    fig.add_trace(go.Scatterpolar(
        r=[R]*len(theta), theta=theta, mode="lines",
        line=dict(color="gray", dash="dot"),
        name="global mean", hoverinfo="skip"), row=1, col=1)

    # within-cluster ±1σ band (the "uncertainty" of the cluster on each dim)
    band_hi = (z_mean + z_std).tolist() + [(z_mean + z_std).iloc[0]]
    band_lo = (z_mean - z_std).tolist() + [(z_mean - z_std).iloc[0]]
    fig.add_trace(go.Scatterpolar(r=[v+R for v in band_hi], theta=theta,
        mode="lines", line=dict(width=0), showlegend=False,
        hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatterpolar(r=[v+R for v in band_lo], theta=theta,
        fill="tonext", fillcolor="rgba(70,130,200,0.25)",
        mode="lines", line=dict(width=0),
        name="±1σ within-cluster"), row=1, col=1)

    # cluster mean polygon — outside ring = above global, inside = below
    means = z_mean.tolist() + [z_mean.iloc[0]]
    fig.add_trace(go.Scatterpolar(r=[v+R for v in means], theta=theta,
        mode="lines+markers",
        line=dict(color="rgb(50,90,180)", width=2),
        name="cluster mean"), row=1, col=1)

    fig.update_polars(radialaxis=dict(range=[0, 2*R], showticklabels=False))

    # signed-z bar chart for the top-N most distinguishing dims
    top = z_mean.head(top_n)
    fig.add_trace(go.Bar(
        x=top.values, y=top.index, orientation="h",
        marker_color=["crimson" if v < 0 else "steelblue" for v in top.values],
        text=[f"within σ={z_std[d]:.2f}" for d in top.index],
        textposition="outside", showlegend=False), row=1, col=2)
    fig.update_xaxes(title="z-score of cluster mean (vs. global)", row=1, col=2)
    fig.update_yaxes(autorange="reversed", row=1, col=2)
    fig.update_layout(
        title=f"Cluster {cluster_id}  •  n={in_cluster.sum()}", height=520)

    # predicate rules — train on ORIGINAL units so thresholds are interpretable
    tree = DecisionTreeClassifier(max_depth=tree_depth,
                                  class_weight="balanced", random_state=0)
    tree.fit(df[feature_cols].values, in_cluster.astype(int).values)
    rules = export_text(tree, feature_names=list(feature_cols))

    return fig, rules

In [15]:
for c in sorted(df["cluster"].unique()):
    if c == -1:
        continue  # skip outliers
    fig, rules = cluster_characteristics(c, df, X_scaled_df, feature_cols)
    fig.show()
    print(f"\n--- Cluster {c} predicates ---\n{rules}")


--- Cluster 0 predicates ---
|--- total sulfur dioxide <= 68.50
|   |--- residual sugar <= 3.77
|   |   |--- total sulfur dioxide <= 52.50
|   |   |   |--- class: 0
|   |   |--- total sulfur dioxide >  52.50
|   |   |   |--- class: 0
|   |--- residual sugar >  3.77
|   |   |--- fixed acidity <= 10.75
|   |   |   |--- class: 1
|   |   |--- fixed acidity >  10.75
|   |   |   |--- class: 0
|--- total sulfur dioxide >  68.50
|   |--- pH <= 3.49
|   |   |--- volatile acidity <= 0.41
|   |   |   |--- class: 1
|   |   |--- volatile acidity >  0.41
|   |   |   |--- class: 1
|   |--- pH >  3.49
|   |   |--- pH <= 3.50
|   |   |   |--- class: 0
|   |   |--- pH >  3.50
|   |   |   |--- class: 0




--- Cluster 1 predicates ---
|--- sulphates <= 0.89
|   |--- pH <= 2.87
|   |   |--- class: 0
|   |--- pH >  2.87
|   |   |--- class: 0
|--- sulphates >  0.89
|   |--- citric acid <= 0.48
|   |   |--- alcohol <= 10.65
|   |   |   |--- class: 1
|   |   |--- alcohol >  10.65
|   |   |   |--- class: 0
|   |--- citric acid >  0.48
|   |   |--- chlorides <= 0.06
|   |   |   |--- class: 0
|   |   |--- chlorides >  0.06
|   |   |   |--- class: 0




--- Cluster 2 predicates ---
|--- free sulfur dioxide <= 15.50
|   |--- total sulfur dioxide <= 44.50
|   |   |--- free sulfur dioxide <= 2.00
|   |   |   |--- class: 0
|   |   |--- free sulfur dioxide >  2.00
|   |   |   |--- class: 0
|   |--- total sulfur dioxide >  44.50
|   |   |--- citric acid <= 0.36
|   |   |   |--- class: 0
|   |   |--- citric acid >  0.36
|   |   |   |--- class: 1
|--- free sulfur dioxide >  15.50
|   |--- volatile acidity <= 0.56
|   |   |--- alcohol <= 10.53
|   |   |   |--- class: 1
|   |   |--- alcohol >  10.53
|   |   |   |--- class: 0
|   |--- volatile acidity >  0.56
|   |   |--- citric acid <= 0.34
|   |   |   |--- class: 0
|   |   |--- citric acid >  0.34
|   |   |   |--- class: 1




--- Cluster 3 predicates ---
|--- chlorides <= 0.24
|   |--- class: 0
|--- chlorides >  0.24
|   |--- pH <= 3.31
|   |   |--- class: 1
|   |--- pH >  3.31
|   |   |--- free sulfur dioxide <= 5.75
|   |   |   |--- class: 0
|   |   |--- free sulfur dioxide >  5.75
|   |   |   |--- class: 0




--- Cluster 4 predicates ---
|--- volatile acidity <= 0.41
|   |--- alcohol <= 10.15
|   |   |--- free sulfur dioxide <= 46.50
|   |   |   |--- class: 0
|   |   |--- free sulfur dioxide >  46.50
|   |   |   |--- class: 1
|   |--- alcohol >  10.15
|   |   |--- fixed acidity <= 10.35
|   |   |   |--- class: 1
|   |   |--- fixed acidity >  10.35
|   |   |   |--- class: 0
|--- volatile acidity >  0.41
|   |--- alcohol <= 11.08
|   |   |--- fixed acidity <= 6.25
|   |   |   |--- class: 0
|   |   |--- fixed acidity >  6.25
|   |   |   |--- class: 0
|   |--- alcohol >  11.08
|   |   |--- citric acid <= 0.27
|   |   |   |--- class: 0
|   |   |--- citric acid >  0.27
|   |   |   |--- class: 1



...


--- Cluster 19 predicates ---
|--- citric acid <= 0.21
|   |--- total sulfur dioxide <= 25.50
|   |   |--- alcohol <= 10.53
|   |   |   |--- class: 1
|   |   |--- alcohol >  10.53
|   |   |   |--- class: 0
|   |--- total sulfur dioxide >  25.50
|   |   |--- sulphates <= 0.50
|   |   |   |--- class: 1
|   |   |--- sulphates >  0.50
|   |   |   |--- class: 0
|--- citric acid >  0.21
|   |--- class: 0




--- Cluster 20 predicates ---
|--- volatile acidity <= 0.51
|   |--- volatile acidity <= 0.50
|   |   |--- free sulfur dioxide <= 1.50
|   |   |   |--- class: 0
|   |   |--- free sulfur dioxide >  1.50
|   |   |   |--- class: 0
|   |--- volatile acidity >  0.50
|   |   |--- total sulfur dioxide <= 14.50
|   |   |   |--- class: 1
|   |   |--- total sulfur dioxide >  14.50
|   |   |   |--- class: 0
|--- volatile acidity >  0.51
|   |--- fixed acidity <= 7.65
|   |   |--- pH <= 3.01
|   |   |   |--- class: 0
|   |   |--- pH >  3.01
|   |   |   |--- class: 0
|   |--- fixed acidity >  7.65
|   |   |--- total sulfur dioxide <= 67.50
|   |   |   |--- class: 1
|   |   |--- total sulfur dioxide >  67.50
|   |   |   |--- class: 0




--- Cluster 21 predicates ---
|--- alcohol <= 10.45
|   |--- density <= 1.00
|   |   |--- total sulfur dioxide <= 50.50
|   |   |   |--- class: 1
|   |   |--- total sulfur dioxide >  50.50
|   |   |   |--- class: 0
|   |--- density >  1.00
|   |   |--- class: 0
|--- alcohol >  10.45
|   |--- class: 0




--- Cluster 22 predicates ---
|--- volatile acidity <= 0.70
|   |--- chlorides <= 0.18
|   |   |--- free sulfur dioxide <= 1.50
|   |   |   |--- class: 0
|   |   |--- free sulfur dioxide >  1.50
|   |   |   |--- class: 0
|   |--- chlorides >  0.18
|   |   |--- citric acid <= 0.19
|   |   |   |--- class: 1
|   |   |--- citric acid >  0.19
|   |   |   |--- class: 0
|--- volatile acidity >  0.70
|   |--- free sulfur dioxide <= 22.50
|   |   |--- sulphates <= 0.68
|   |   |   |--- class: 1
|   |   |--- sulphates >  0.68
|   |   |   |--- class: 0
|   |--- free sulfur dioxide >  22.50
|   |   |--- pH <= 3.11
|   |   |   |--- class: 0
|   |   |--- pH >  3.11
|   |   |   |--- class: 0




--- Cluster 23 predicates ---
|--- citric acid <= 0.16
|   |--- alcohol <= 10.05
|   |   |--- class: 0
|   |--- alcohol >  10.05
|   |   |--- residual sugar <= 2.45
|   |   |   |--- class: 0
|   |   |--- residual sugar >  2.45
|   |   |   |--- class: 1
|--- citric acid >  0.16
|   |--- volatile acidity <= 0.91
|   |   |--- chlorides <= 0.02
|   |   |   |--- class: 0
|   |   |--- chlorides >  0.02
|   |   |   |--- class: 0
|   |--- volatile acidity >  0.91
|   |   |--- residual sugar <= 4.10
|   |   |   |--- class: 0
|   |   |--- residual sugar >  4.10
|   |   |   |--- class: 1

